# US-02: Pipeline de limpieza de datos
# Este notebook completa la limpieza sin modificar el código de ingesta existente.

In [1]:
import pandas as pd
import numpy as np

# Si estás en Colab y necesitas subir el archivo:
# from google.colab import files
# files.upload()

In [3]:
# Cargar el dataset
df = pd.read_excel("Telco_customer_churn.xlsx")  # en Colab
# df = pd.read_excel("../data/raw/Telco_customer_churn.xlsx")  # en local

print(f"Dataset original: {df.shape}")
df.head()

Dataset original: (7043, 33)


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [4]:
# 1. Convertir Total Charges a numérico y rellenar nulos
df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce")
nulos = df["Total Charges"].isnull().sum()
df["Total Charges"] = df["Total Charges"].fillna(df["Monthly Charges"] * df["Tenure Months"])
print(f"Se corrigieron {nulos} nulos en Total Charges")

Nulos corregidos en Total Charges: 11


In [5]:
# 2. Eliminar columnas que no sirven para el modelo
columnas_eliminar = [
    "CustomerID", "Count", "Country", "State", "City", "Zip Code",
    "Lat Long", "Latitude", "Longitude", "Churn Label",
    "Churn Score", "CLTV", "Churn Reason"
]
columnas_eliminar = [c for c in columnas_eliminar if c in df.columns]
df = df.drop(columns=columnas_eliminar)
print(f"Columnas eliminadas: {len(columnas_eliminar)}")
print(f"Shape actual: {df.shape}")

Columnas eliminadas: 13
Shape actual: (7043, 20)


In [7]:
# 3. Convertir variables binarias Yes/No a 1/0
binarias = ["Senior Citizen", "Partner", "Dependents", "Phone Service", "Paperless Billing"]
for col in binarias:
    if col in df.columns:
        df[col] = df[col].map({"Yes": 1, "No": 0}).fillna(0).astype(int)

In [8]:
# 4. One-hot encoding de variables categóricas
categoricas = [
    "Multiple Lines", "Internet Service", "Online Security", "Online Backup",
    "Device Protection", "Tech Support", "Streaming TV", "Streaming Movies",
    "Contract", "Payment Method", "Gender"
]
categoricas = [c for c in categoricas if c in df.columns]
df = pd.get_dummies(df, columns=categoricas, drop_first=True)

print(f"Shape final: {df.shape}")
print(f"Nulos restantes: {df.isnull().sum().sum()}")

Shape final: (7043, 31)
Nulos restantes: 0


In [9]:
# 5. Separar X e y
y = df["Churn Value"]
X = df.drop(columns=["Churn Value"])

print("Datos listos para modelar")
print(f"X: {X.shape[0]} filas y {X.shape[1]} columnas")
print(f"y: {y.shape[0]} registros")
print("\nDistribución del target:")
print(y.value_counts(normalize=True).round(3))
print("\nCon esto ya se puede pasar a entrenar los modelos (US-05 y US-06).")

X: (7043, 30)
y: (7043,)

Distribución del target:
Churn Value
0    0.735
1    0.265
Name: proportion, dtype: float64


In [11]:
# 6. Guardar el dataset limpio
df.to_parquet("telco_limpio.parquet", index=False)
print("Dataset limpio guardado como telco_limpio.parquet")

Dataset limpio guardado como telco_limpio.parquet
